In [1]:
from pathlib import Path
import xmltodict
import pandas as pd
import numpy as np
import tifffile as tiff
import xmltodict
from dateutil import parser
import re

import matplotlib.pyplot as plt
from collections import namedtuple
from shapely.geometry import box

from skimage.registration import phase_cross_correlation

In [8]:
from atlas.io import extract_tif_metadata, extract_s_number, get_pixel_size_from_tif, get_image_size_from_tif
from atlas.image_analysis import image_dtype_min_max, mask_low_and_saturation

In [9]:
def normalize_angle(angle):
    """
    Normalizes an angle to the range [-180, 180] degrees.

    Parameters:
    ----------
    angle : float
        The input angle in degrees.

    Returns:
    -------
    float
        The normalized angle within [-180, 180] range.
    """
    return ((angle + 180) % 360) - 180

def rotate_points(points, angle_degrees):
    """
    Rotates one or multiple 2D points (x, y) by a given angle in degrees.

    Parameters:
    ----------
    points : tuple (x, y) or list of tuples [(x1, y1), (x2, y2), ...]
        The original point(s) to be rotated.
    angle_degrees : float
        The rotation angle in degrees.

    Returns:
    -------
    tuple (x', y') or list of tuples [(x1', y1'), (x2', y2'), ...]
        The rotated point(s).
    """
    # Convert input to NumPy array
    points_array = np.array(points, dtype=np.float64)  # Ensure it's float for precision

    # If a single point was given, reshape it to (1, 2)
    if points_array.ndim == 1:
        points_array = points_array.reshape(1, 2)

    # Convert angle to radians
    angle_radians = np.radians(angle_degrees)

    # Define the rotation matrix
    rotation_matrix = np.array([
        [np.cos(angle_radians), -np.sin(angle_radians)],
        [np.sin(angle_radians),  np.cos(angle_radians)]
    ])

    # Apply rotation (matrix multiplication)
    rotated_points = points_array @ rotation_matrix.T  # Transpose for correct multiplication

    # Convert back to original format (tuple or list of tuples)
    if len(rotated_points) == 1:
        return tuple(rotated_points[0])  # Return a single tuple for single input
    return [tuple(point) for point in rotated_points]  # Return a list of tuples for multiple points

# Define NamedTuple for Rectangle
Rectangle = namedtuple("Rectangle", ["top", "bot", "left", "right"])

# Function to create both Rectangle and Shapely box
def create_geometry(row):
    """
    Creates a Rectangle namedtuple and a Shapely box geometry from a row in the DataFrame.

    Parameters:
    ----------
    row : pandas.Series
        A row from the DataFrame.

    Returns:
    -------
    tuple(Rectangle, shapely.geometry.box)
        A tuple containing the namedtuple Rectangle and the Shapely box object.
    """
    rect = Rectangle(top=row['Y0_pix'], bot=row['Y1_pix'], left=row['X0_pix'], right=row['X1_pix'])
    geom = box(minx=rect.left, maxx=rect.right, miny=rect.top, maxy=rect.bot)
    return rect, geom

def get_overlap_relative(box_reference, box_moving):
    # Compute the intersection
    overlap = box_reference.intersection(box_moving)

    # Convert the overlapping box to integer pixel indices
    min_x, min_y, max_x, max_y = map(int, overlap.bounds)

    # overlap is in final image space (boxes reference frame), 
    # now I have to change it to the pixel space of each tif.
    img0_x_0 = min_x - int(box_reference.bounds[0])
    img0_x_1 = max_x - int(box_reference.bounds[0])
    img0_y_0 = min_y - int(box_reference.bounds[1])
    img0_y_1 = max_y - int(box_reference.bounds[1])

    ref_overlap = box(minx=img0_x_0, miny=img0_y_0, maxx=img0_x_1, maxy=img0_y_1)
    print(f"overlap img0 x: {img0_x_0}-{img0_x_1}, y {img0_y_0}-{img0_y_1}")


    img1_x_0 = min_x - int(box_moving.bounds[0])
    img1_x_1 = max_x - int(box_moving.bounds[0])
    img1_y_0 = min_y - int(box_moving.bounds[1])
    img1_y_1 = max_y - int(box_moving.bounds[1])
    mov_overlap = box(minx=img1_x_0, miny=img1_y_0, maxx=img1_x_1, maxy=img1_y_1)
    print(f"overlap img1 x: {img1_x_0}-{img1_x_1}, y {img1_y_0}-{img1_y_1}")

    return ref_overlap, mov_overlap

def get_tiles_dataframe(mif_file, buffer_microns):
    # loads mif file and parces it into a dictionary, takes care of sorting the tiles by
    # acquisition time and also takes into account, scan rotation and buffer.

    raw_data_folder = mif_file.parent
    with open(mif_file, "r", encoding="utf-8") as f:
        mif_dict = xmltodict.parse(f.read())
    
    # Extract tile list (or single dictionary)
    mif_tile_list = mif_dict['MosaicInfo']['Tiles']['Tile']

    # Ensure mif_tile_list is always a list
    if isinstance(mif_tile_list, dict):  # If it's a single dictionary, convert to a list
        mif_tile_list = [mif_tile_list]

    # Convert to DataFrame
    mif_tile_df = pd.DataFrame(mif_tile_list)

        # Convert all timestamps in 'StartTime' column to datetime objects
    mif_tile_df['StartTime'] = mif_tile_df['StartTime'].apply(parser.isoparse)

    mif_tile_df['ScanRotationDeg'] = normalize_angle(float(mif_dict['MosaicInfo']['ReferenceInfo']['Beam']['ScanRot']))

    # get pixel size to each row in the DataFrame
    mif_tile_df['PixelSizeMicron'] = mif_tile_df['Filename'].apply(
        lambda fname: get_pixel_size_from_tif(fname, raw_data_folder))
    # loads image size based on tif metadata, we do not open the pixel info
    mif_tile_df[['ImageWidth', 'ImageHeight']] = mif_tile_df['Filename'].apply(
        lambda fname: get_image_size_from_tif(fname, raw_data_folder)
    ).to_list()
    # Convert columns to float
    mif_tile_df['StageX'] = pd.to_numeric(mif_tile_df['StageX'], errors='coerce')
    mif_tile_df['StageY'] = pd.to_numeric(mif_tile_df['StageY'], errors='coerce')

    # Apply rotation to each row, this is important to keep the square exports of FIBICs in the correct frame of ref
    mif_tile_df[['StageX_rot', 'StageY_rot']] = mif_tile_df.apply(
        lambda row: pd.Series(rotate_points((row['StageX'], row['StageY']), row['ScanRotationDeg'] * -1)),
        axis=1  # Apply function row-wise
    )

    # now we set the reference frame to 0,0 and not to the actual stage position
    min_stage_X = mif_tile_df['StageX_rot'].min()
    min_stage_Y = mif_tile_df['StageY_rot'].min()

    # buffer is needed for later stitching, if not some images might go out of the pre alocated canvas
    buffer_pixels = int(np.round(buffer_microns/mif_tile_df['PixelSizeMicron'][0]))
    buffer_microns = buffer_pixels * mif_tile_df['PixelSizeMicron'][0]

    mif_tile_df['X0_micron'] = mif_tile_df['StageX_rot']-min_stage_X + buffer_microns
    mif_tile_df['Y0_micron'] = mif_tile_df['StageY_rot']-min_stage_Y + buffer_microns
    mif_tile_df['X0_pix'] = np.round(mif_tile_df['X0_micron']/mif_tile_df['PixelSizeMicron']).astype('uint')
    mif_tile_df['X1_pix'] = mif_tile_df['X0_pix'] + mif_tile_df['ImageWidth']
    mif_tile_df['Y0_pix'] = np.round(mif_tile_df['Y0_micron']/mif_tile_df['PixelSizeMicron']).astype('uint')
    mif_tile_df['Y1_pix'] = mif_tile_df['Y0_pix'] + mif_tile_df['ImageHeight']

    # I sort by time due to my simple tiling strategy later on
    mif_tile_df.sort_values('StartTime', ascending=True, inplace=True)

    # Apply geometry function to all rows and create new columns, probably geometry is enough, check later
    mif_tile_df[['rectangle', 'geometry']] = mif_tile_df.apply(lambda row: pd.Series(create_geometry(row)), axis=1)
    # during stitching I will modify the geomtry to account for the iamge shifts
    mif_tile_df['geometry_shifted'] = mif_tile_df['geometry'].copy()

    return mif_tile_df

def get_total_canvas_size(tile_df):
    # based on the tiles dataframe returns the total size of the canvas needed for stitching
    buffer_pixels = tile_df['X0_pix'].min()
    print(f"buffer in pixels based on DF output: {buffer_pixels}")

    # here we generate the full canvas size
    max_X_row = tile_df.loc[tile_df["X0_pix"].idxmax()]
    max_Y_row = tile_df.loc[tile_df["Y0_pix"].idxmax()]
    total_img_width = max_X_row['X0_pix'] + max_X_row['ImageWidth'] + buffer_pixels
    total_img_height = max_Y_row['Y0_pix'] + max_Y_row['ImageHeight'] + buffer_pixels

    print(f"Total image will be of size including buffer: {total_img_width}x{total_img_height}")

    return total_img_width, total_img_height

def stitch_ATLAS_tiles(tiles_df, raw_data_folder, max_shift_pixels=100):
    # Step 1: Initialize the full canvas and add the first image (reference)
    first_tif_path = raw_data_folder.joinpath(Path(tiles_df.iloc[0]['Filename']).name)
    with tiff.TiffFile(first_tif_path) as tif:
        image_dtype = tif.pages[0].dtype  # Read dtype from metadata


    total_img_width, total_img_height = get_total_canvas_size(tiles_df)

    # Initialize full canvas
    stitched_img = np.zeros([total_img_height, total_img_width], dtype=image_dtype)

    # First image: Never shifted, it's the reference
    row0 = tiles_df.iloc[0]
    box0 = row0['geometry']
    tif_0 = raw_data_folder.joinpath(Path(row0['Filename']).name)
    img0 = np.flipud(tiff.imread(tif_0))  # Flip image

    # Place reference image in full image
    y0, y1 = int(box0.bounds[1]), int(box0.bounds[3])
    x0, x1 = int(box0.bounds[0]), int(box0.bounds[2])
    stitched_img[y0:y1, x0:x1] = img0

    # Step 2: Iterate over all remaining images and align them
    for moving_index in range(1, len(tiles_df)):
        print(f"\nProcessing tile {moving_index}/{len(tiles_df) - 1}...")

        # Reference tile (previous row)
        row_ref = tiles_df.iloc[moving_index - 1]
        box_ref = row_ref['geometry_shifted']
        tif_ref = raw_data_folder.joinpath(Path(row_ref['Filename']).name)
        #w_ref = row_ref['ImageWidth']
        h_ref = row_ref['ImageHeight']

        # Moving tile (current row)
        row_mov = tiles_df.iloc[moving_index]
        box_mov = row_mov['geometry_shifted']
        tif_mov = raw_data_folder.joinpath(Path(row_mov['Filename']).name)
        img_mov = np.flipud(tiff.imread(tif_mov))  # Flip image

        # Compute the intersection area between reference and moving image
        ref_box, mov_box = get_overlap_relative(box_reference=box_ref, box_moving=box_mov)

        # Load only the overlapping part of the reference image
        with tiff.TiffFile(tif_ref) as tif:
            #due to flip
            y0_tmp,y1_tmp = int(ref_box.bounds[1]), int(ref_box.bounds[3])
            y0 = h_ref - y1_tmp
            y1 = h_ref - y0_tmp
            x0,x1 = int(ref_box.bounds[0]), int(ref_box.bounds[2])
            crop_ref = tif.asarray()[y0:y1, x0:x1]
            crop_ref = np.flipud(crop_ref)

        # Extract the overlapping part of the moving image
        crop_mov = img_mov[int(mov_box.bounds[1]):int(mov_box.bounds[3]),
                        int(mov_box.bounds[0]):int(mov_box.bounds[2])]

        # Create masks for saturation and low values
        mask_ref = np.logical_not(mask_low_and_saturation(crop_ref))
        mask_mov = np.logical_not(mask_low_and_saturation(crop_mov))

        # Compute phase cross-correlation shift
        detected_shift, _, _ = phase_cross_correlation(crop_ref, crop_mov, reference_mask=mask_ref, moving_mask=mask_mov, return_error='always')

        print(f"Detected pixel offset (row, col): {-detected_shift}")

        if any(np.abs(detected_shift) > max_shift_pixels):
            print('something weird, setting offset to 0,0')
            detected_shift[0] = 0
            detected_shift[1] = 0

        # Apply shift to position the moving image correctly
        y0 = int(box_mov.bounds[1] + detected_shift[0])
        y1 = int(box_mov.bounds[3] + detected_shift[0])
        x0 = int(box_mov.bounds[0] + detected_shift[1])
        x1 = int(box_mov.bounds[2] + detected_shift[1])

        # Insert the moved image into the full image
        stitched_img[y0:y1, x0:x1] = img_mov

        # Update the geometry_shifted column
        shifted_box = box(minx=x0, miny=y0, maxx=x1, maxy=y1)
        tiles_df.loc[moving_index, 'geometry_shifted'] = shifted_box

    stitched_img = np.flipud(stitched_img)

    print("\n✅ Stitching process completed. now saving")
    extracted_number = extract_s_number(first_tif_path)

    # Define the output file path
    output_tif_path = raw_data_folder.parent.joinpath(f"stitched_image_S_{extracted_number}.tiff")
    output_cc_path = raw_data_folder.parent.joinpath(f"phaseCC_stitching_S_{extracted_number}.csv")
    # Save the full image as a TIFF file
    tiff.imwrite(output_tif_path, stitched_img)

    tiles_df.to_csv(output_cc_path, index=False)

    print("saving done!")

    return stitched_img, tiles_df

# TODO: get a helper function that can stitch based on the csv output, 
# the current method is memory efficient because I have to load the iamges
# however I could just load the overlap region in the moving image as I do
# for the ref. This would split efforts and could perhaps be easier to maintain

In [10]:
series_folder = Path(r"C:\Users\xcamra\Documents\pybias-prototype\ATLAS\data\test-series")

series_list = []
for folder in series_folder.iterdir():  # Iterate over all items in the folder
    if folder.is_dir() and folder.name.startswith("S_"):
        print(f"fould series folder: {folder.name}")
        series_list.append(folder)

for raw_data_folder in series_list:
    # Check if files in the folder have the ".ve-tie" extension
    tie_file = None
    mif_file = None
    for file in raw_data_folder.iterdir():  # Iterate over all items in the folder
        #if file.is_file() and file.suffix == ".ve-tie":  # Check if it's a file with the desired extension
        #    print(f"File with '.ve-tie' extension found: {file.name}")
        #    tie_file = file
        if file.is_file() and file.suffix == ".ve-mif":  # Check if it's a file with the desired extension
            print(f"File with '.ve-mif' extension found: {file.name}")
            mif_file = file

            mif_tile_df = get_tiles_dataframe(mif_file, buffer_microns=10)

            #total_img_width, total_img_height = get_total_canvas_size(mif_tile_df)

            stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, raw_data_folder, max_shift_pixels=200)

            #fig, ax = plt.subplots(1, 1, figsize=(15, 15))
            #ax.imshow(stitched_img[:,:], cmap='gray')


fould series folder: S_001_883976504
fould series folder: S_002_192360474
fould series folder: S_003_117038215
File with '.ve-mif' extension found: MosaicInfo_S_001_883976504.ve-mif
buffer in pixels based on DF output: 500
Total image will be of size including buffer: 14579x8000

Processing tile 1/1...
overlap img0 x: 6579-7000, y 0-7000
overlap img1 x: 0-421, y 0-7000
Detected pixel offset (row, col): [ 14. -76.]

✅ Stitching process completed. now saving
saving done!
File with '.ve-mif' extension found: MosaicInfo_S_002_192360474.ve-mif
buffer in pixels based on DF output: 500
Total image will be of size including buffer: 14579x8000

Processing tile 1/1...
overlap img0 x: 6579-7000, y 0-7000
overlap img1 x: 0-421, y 0-7000
Detected pixel offset (row, col): [  17. -119.]

✅ Stitching process completed. now saving
saving done!
File with '.ve-mif' extension found: MosaicInfo_S_003_117038215.ve-mif
buffer in pixels based on DF output: 500
Total image will be of size including buffer: 145

In [ ]:
# omit below this is if stitching fails badly, so I trust stage positionsi want to stitch one at the time
raise "out of here"

In [ ]:
#raw_data_folder = Path(r"Y:\Andre-nano-sims\2025-Jan-batch-01\Region105_1126825830")
#raw_data_folder = Path(r"Y:\LUKE\ATLAS-projects\Lenka-TA19-22_data\session_385872141\TA19-100nm\S_001_2044144885")
#raw_data_folder = Path(r"Y:\LUKE\ATLAS-projects\Lenka-TA19-22_data\session_385872141\TA19-100nm\S_002_1880554192")
#raw_data_folder = Path(r"Y:\LUKE\ATLAS-projects\Lenka-TA19-22_data\session_385872141\TA19-100nm\S_003_1749985812")
#raw_data_folder = Path(r"Y:\LUKE\ATLAS-projects\Lenka-TA19-22_data\session_385872141\TA19-100nm\S_004_367552619")
#raw_data_folder = Path(r"C:\Users\xcamra\Documents\pybias-prototype\ATLAS\data\test-series\S_001_883976504")
raw_data_folder = Path(r"C:\Users\xcamra\Documents\pybias-prototype\ATLAS\data\test-series\S_002_192360474")
#raw_data_folder = Path(r"C:\Users\xcamra\Documents\pybias-prototype\ATLAS\data\test-series\S_003_117038215")

#raw_data_folder = Path(r"Y:\LUKE\ATLAS-projects\Lenka-TA19-22_data\session_385872141\TA20-roi20p1\S_001_1028563958")



# Check if files in the folder have the ".ve-tie" extension
tie_file = None
mif_file = None
for file in raw_data_folder.iterdir():  # Iterate over all items in the folder
    #if file.is_file() and file.suffix == ".ve-tie":  # Check if it's a file with the desired extension
    #    print(f"File with '.ve-tie' extension found: {file.name}")
    #    tie_file = file
    if file.is_file() and file.suffix == ".ve-mif":  # Check if it's a file with the desired extension
        print(f"File with '.ve-mif' extension found: {file.name}")
        mif_file = file

        mif_tile_df = get_tiles_dataframe(mif_file, buffer_microns=10)

        #total_img_width, total_img_height = get_total_canvas_size(mif_tile_df)

        stitched_img, mif_tile_df = stitch_ATLAS_tiles(mif_tile_df, max_shift_pixels=200)

        #fig, ax = plt.subplots(1, 1, figsize=(15, 15))
        #ax.imshow(stitched_img[:,:], cmap='gray')




In [ ]:
# omit below this is if stitching fails badly, so I trust stage positions
raise "out of here"

In [ ]:
# Read image dtype from the first file (without loading data)
first_tif_path = raw_data_folder.joinpath(Path(mif_tile_df.iloc[0]['Filename']).name)
with tiff.TiffFile(first_tif_path) as tif:
    image_dtype = tif.pages[0].dtype  # Read dtype from metadata

# Initialize the full image canvas with zeros
full_img = np.zeros([total_img_height, total_img_width], dtype=image_dtype)

# Iterate over all rows in the DataFrame
for idx, row in mif_tile_df.iterrows():
    # Extract filename and create full path
    tif_path = raw_data_folder.joinpath(Path(row['Filename']).name)

    print(f"Processing {tif_path.name}")

    # Flip image due to coordinate system differences
    image_array = np.flipud(tiff.imread(tif_path))

    # Print shape and type
    print("Image shape:", image_array.shape)
    print("Data type:", image_array.dtype)

    # Extract pixel coordinates
    x0, y0 = row['X0_pix'], row['Y0_pix']
    x1, y1 = x0 + row['ImageWidth'], y0 + row['ImageHeight']

    # Place the image into the full image
    full_img[y0:y1, x0:x1] = image_array

full_img = np.flipud(full_img)

print("Stitching complete.")

# Define the output file path
output_tif_path = raw_data_folder.joinpath("stage_stitched_image.tiff")

# Save the full image as a TIFF file
tiff.imwrite(output_tif_path, full_img)

print(f"Saved stitched image to: {output_tif_path}")


plt.imshow(full_img)
